# Notebook 3 – Train, Validation & Test

## Dataset - (`data.csv`)


In [1]:
import pandas as pd
df = pd.read_csv('data.csv', encoding='latin1')
df = df.dropna(subset=['CustomerID'])
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34


## 1. Training Dataset

The **Training Dataset** is the portion of data the model actually learns from. The model looks at the features and the correct answers in this set, and adjusts itself to capture the relationship between them.

Think of it like a student studying from a textbook with worked examples — this is where the actual learning happens.

## 2. Validation Dataset

The **Validation Dataset** is a separate slice of data used *during development* to check how well the model is doing, and to make decisions — like which hyperparameters to use, or which model to pick.

It's like practice tests a student takes while studying — useful for guiding what to improve, but not the final exam.

## 3. Test Dataset

The **Test Dataset** is a completely separate slice of data, kept aside and touched only once, at the very end. It gives an honest, unbiased estimate of how the model will perform on new, real-world data.

This is the final exam — it should never be used to guide any decisions along the way (see the discussion on this above).

## 4. Train/Test Split

**Train/Test Split** is the simplest way to divide data: split it into a training portion and a testing portion, usually 70-30 or 80-20.

In [2]:
from sklearn.model_selection import train_test_split
X = df[['Quantity', 'UnitPrice']]
y = df['TotalPrice']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 318307
Test size: 79577


## 5. Train/Validation/Test Split

For most real projects, we split data into **three** parts instead of two — training, validation, and test. A common ratio is 60% train, 20% validation, 20% test.

This is done in two steps: first separate out the test set, then split what's left into training and validation.

In [11]:
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)
print("Train:", len(X_train), " Validation:", len(X_val), " Test:", len(X_test))

Train: 238730  Validation: 79577  Test: 79577


## 6. Cross Validation

**Cross Validation** is a way to check a model's performance more reliably than a single train/validation split. Instead of one split, the data is split multiple times in different ways, and the model's performance is averaged across all of them.

This gives a more trustworthy estimate, especially when the dataset is small — a single lucky or unlucky split can otherwise be misleading.

## 7. K-Fold Cross Validation

**K-Fold Cross Validation** is the most common form of cross validation. The data is divided into `K` equal parts ("folds"). The model is trained on `K-1` folds and tested on the remaining fold — this is repeated `K` times, so every fold gets used as the test set exactly once. The final score is the average across all `K` runs.

In [10]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
model = LinearRegression()
scores = cross_val_score(model, X, y, cv=5)  # 5-Fold 
print("Scores for each fold:", scores)
print("Average score:", scores.mean())

Scores for each fold: [0.13935994 0.13204074 0.30139103 0.44531252 0.74980117]
Average score: 0.35358107837476915


## 8. Stratified K-Fold

**Stratified K-Fold** is a version of K-Fold used for classification problems, where each fold keeps the same proportion of each class as the full dataset.

This matters when classes are imbalanced — for example, if only 5% of transactions are "high value," a regular K-Fold split might accidentally put very few (or zero) high-value examples in some folds. Stratified K-Fold prevents that.

In [9]:
from sklearn.model_selection import StratifiedKFold
df['ValueClass'] = (df['TotalPrice'] > df['TotalPrice'].median()).astype(int)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold, (train_idx, val_idx) in enumerate(skf.split(df, df['ValueClass'])):
    print(f"Fold {fold}: train={len(train_idx)}, validation={len(val_idx)}")

Fold 0: train=318307, validation=79577
Fold 1: train=318307, validation=79577
Fold 2: train=318307, validation=79577
Fold 3: train=318307, validation=79577
Fold 4: train=318308, validation=79576


## 9. Random State

**Random State** is a fixed number used to control randomness, so that operations like shuffling or splitting produce the **same result every time** they're run.

This matters for reproducibility — without a fixed random state, you'd get a slightly different split every time you re-run the code, making it hard to compare results fairly or debug issues.

In [8]:
split_a = train_test_split(X, y, test_size=0.2, random_state=42)
split_b = train_test_split(X, y, test_size=0.2, random_state=42)
(split_a[0].equals(split_b[0]))

True

## 10. Data Leakage

**Data Leakage** happens when information from outside the training set — often information that wouldn't be available at real prediction time — accidentally influences the model. This makes the model look much better during testing than it actually will be in production.

**Common examples:**
- Scaling or preprocessing the *entire* dataset before splitting into train/test (the training process "sees" statistics from the test set)
- Including a feature that's only known after the outcome has already happened (e.g. using a "refund issued" column to predict fraud)
- Duplicate rows appearing in both the training and test sets

**Rule of thumb:** always split your data first, and fit any preprocessing (scalers, encoders, etc.) only on the training set.

## 11. Overfitting

**Overfitting** happens when a model learns the training data *too well* — including its noise and random quirks — instead of learning the general pattern. It performs great on training data, but poorly on new, unseen data.

Think of a student who memorizes answers to specific practice questions instead of understanding the underlying concept — they'll fail as soon as the exam questions are phrased differently.

**Signs of overfitting:** very high training accuracy, but noticeably lower validation/test accuracy.

## 12. Underfitting

**Underfitting** is the opposite problem — the model is too simple to capture the real pattern in the data at all. It performs poorly on both the training data and the test data.

Think of a student who barely studied — they'll do badly on both the practice questions and the real exam.

**Signs of underfitting:** low accuracy on both training and validation/test sets.